# 6. Воспроизводимый отчёт

**Цель:** выполнить поддерживаемый `GraphEVTPipeline` на Kuramoto-ряде и сохранить численный результат, manifest и краткий отчёт, включая граф и ranking, когда они доступны.

In [1]:
from pathlib import Path
from importlib.metadata import version
from hashlib import sha256
import json
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/kuramoto_synchronized_series.csv").exists())
DATA = ROOT / "data/generated/kuramoto_synchronized_series.csv"
SEED = 42
df = pd.read_csv(DATA)
channel_names = [column for column in df.columns if column.startswith("channel_")]
x = df[channel_names].to_numpy()
assert np.isfinite(x).all() and x.ndim == 2

In [2]:
BASELINE_SIZE = 300
parameters = {
    "baseline_size": BASELINE_SIZE,
    "tail_quantile": 0.90,
    "alarm_probability": 0.99,
    "persistence": 2,
    "top_k": 3,
    "graph_method": "ar",
}
evt = EVTConfig(**{k: v for k, v in parameters.items() if k != "graph_method"})
result = GraphEVTPipeline(
    evt,
    graph=GraphConfig(method=parameters["graph_method"], edge_threshold=0.35, random_state=SEED),
    input_config=InputConfig(missing="error", require_regular_time=True),
    verbose=True,
).run_detailed(x)
detection = result.detection
estimated_source = None if result.ranking is None else channel_names[result.ranking.source]

artifact_dir = ROOT / "results/n-d/06_report"
artifact_dir.mkdir(parents=True, exist_ok=True)
indicator_path = artifact_dir / "indicator.csv"
pd.DataFrame({
    "time": df["timestamp"],
    "indicator": detection.indicator,
    "above_alarm_threshold": detection.indicator > detection.alarm_threshold,
    "is_baseline": np.arange(len(x)) < BASELINE_SIZE,
}).to_csv(indicator_path, index=False)

Graph-EVT pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

In [3]:
manifest = {
    "data_file": str(DATA.relative_to(ROOT)),
    "data_sha256": sha256(DATA.read_bytes()).hexdigest(),
    "library": "graph-evt-agent",
    "library_version": version("graph-evt-agent"),
    "seed": SEED,
    "parameters": parameters,
    "input_profile": {
        "kind": result.input_profile.kind,
        "n_time": result.input_profile.n_time,
        "n_channels": result.input_profile.n_channels,
    },
    "detection": {
        "detected": detection.detected,
        "time_index": detection.time_index,
        "alarm_threshold": float(detection.alarm_threshold),
    },
    "graph_available": result.graph is not None,
    "estimated_source": estimated_source,
    "artifacts": {"indicator_csv": indicator_path.name},
    "limitations": [
        "synthetic unlabelled multichannel series",
        "graph edges are estimated on the baseline only, not causal",
        "source ranking is an explainable heuristic, not a trained causal model",
        "results depend on baseline, EVT configuration, and graph method",
    ],
}
with (artifact_dir / "manifest.json").open("w", encoding="utf-8") as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

status = "обнаружено" if detection.detected else "не обнаружено"
summary = f"""# Краткий отчёт

На {len(x)} наблюдениях по {result.input_profile.n_channels} каналам устойчивое превышение {status}.
Alarm-порог: {detection.alarm_threshold:.6g}; первый индекс: {detection.time_index}; оценённый источник: {estimated_source}.
Численный ряд indicator сохранён в `indicator.csv`, параметры и ограничения — в `manifest.json`.
"""
(artifact_dir / "report.md").write_text(summary, encoding="utf-8")
manifest

{'data_file': 'data/generated/kuramoto_synchronized_series.csv',
 'data_sha256': '6b44361d5df20571b49c0cb1ea9239c614d9c4cb178fbd606f83ab1f2b1c2564',
 'library': 'graph-evt-agent',
 'library_version': '0.1.0',
 'seed': 42,
 'parameters': {'baseline_size': 300,
  'tail_quantile': 0.9,
  'alarm_probability': 0.99,
  'persistence': 2,
  'top_k': 3,
  'graph_method': 'ar'},
 'input_profile': {'kind': 'multivariate', 'n_time': 1500, 'n_channels': 12},
 'detection': {'detected': True,
  'time_index': 300,
  'alarm_threshold': 5.494979693764073},
 'graph_available': True,
 'estimated_source': 'channel_08',
 'artifacts': {'indicator_csv': 'indicator.csv'},
 'limitations': ['synthetic unlabelled multichannel series',
  'graph edges are estimated on the baseline only, not causal',
  'source ranking is an explainable heuristic, not a trained causal model',
  'results depend on baseline, EVT configuration, and graph method']}

In [4]:
required = {"data_sha256", "library_version", "parameters", "input_profile", "detection", "artifacts", "limitations"}
missing = required - set(manifest)
assert not missing, f"Неполный manifest: {sorted(missing)}"
assert len(manifest["data_sha256"]) == 64
summary

'# Краткий отчёт\n\nНа 1500 наблюдениях по 12 каналам устойчивое превышение обнаружено.\nAlarm-порог: 5.49498; первый индекс: 300; оценённый источник: channel_08.\nЧисленный ряд indicator сохранён в `indicator.csv`, параметры и ограничения — в `manifest.json`.\n'

Повторный запуск при той же версии библиотеки и тех же входных байтах должен воспроизводить численный результат, включая граф и ranking.

**Самопроверка:** зачем в manifest хранится метод графа отдельно от EVT-параметров? Почему `estimated_source` относится к ограничениям, а не к подтверждённым фактам? Какие выводы нельзя переносить за пределы этого ряда?